# External Database Validation

This notebook cross-checks the lncRNA genes of interest (`test_genes`) against two independent, external data sources, separate from the PASNet/SHAP/Driver/SJARACNe pipeline outputs used in the previous notebooks:

1. **lncHUB2 UMAP embedding** — a 2-D UMAP projection of lncRNA co-expression/functional profiles, used here to test whether the genes of interest sit unusually close (in embedding space) to a reference set of already-validated lncRNAs, via a permutation test.
2. **lncHUB2 tissue and cell-line expression tables** — per-tissue and per-cell-line expression summary statistics, visualized as boxplots for the genes of interest.

**What the notebook produces, step by step:**
1. Load the lncHUB2 UMAP coordinate table
2. Run a permutation test comparing each test gene's UMAP-space distance to a known/validated gene set against a null distribution of random genes
3. Plot the UMAP embedding, highlighting the known and test genes
4. Load the tissue- and cell-line-level expression tables
5. Plot per-gene expression distributions across the top tissues and top cell lines

> **Note:** Paths are relative to a `data/lncHUB2/` directory two levels above this notebook's folder, matching the pipeline's external-data layout.


## 1. Imports


In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist
import umap
import matplotlib.pyplot as plt
from adjustText import adjust_text

## 2. lncHUB2 Database — UMAP Embedding Proximity


### 2.1 Load UMAP Coordinates

`umap_results_df.csv` contains, for every gene in the lncHUB2 database, its 2-D UMAP coordinates (`x`, `y`). The gene identifier column is unnamed in the source file, so it's renamed to `gene` for clarity.


In [ ]:
# Load the lncHUB2 UMAP coordinate table.
df = pd.read_csv("../../data/lncHUB2/umap_results_df.csv")

# The gene-identifier column has no header in the source file; name it explicitly.
df = df.rename(columns={"Unnamed: 0": "gene"})
print(df.head())


### 2.2 Permutation Test: UMAP-Space Proximity to a Known Gene Set

This cell tests, for each gene in `test_genes`, whether its position in the UMAP embedding is closer to the `known_genes` reference set than would be expected by chance.

Mechanics:
1. `avg_distance_to_set` computes, for a set of query coordinates, each point's nearest-neighbor Euclidean distance to the target coordinate set, then averages those nearest-neighbor distances.
2. The **observed distance** is this average computed between a given test gene's coordinates and the known-gene coordinates.
3. A **null distribution** is built by repeating the same calculation `n_permutations` times, each time substituting a random sample of points (matched in count to the test gene's points) drawn from the full coordinate table in place of the test gene.
4. The **p-value** is the fraction of permuted (random) distances that are as small as, or smaller than, the observed distance — i.e. how often a random gene would be at least as close to the known set as the real test gene is.


In [ ]:
# --- Gene sets ---
known_genes = ["DLEU1", "DLEU2", "MIAT", "NEAT1", "PVT1"]  # reference lncRNA gene set

test_genes = ["FIRRE", "LINC00158", "KCNJ2-AS1", "FLNB-AS1"]  # lncRNAs of interest

# For each row in gene_coords, find its nearest point in target_coords, then
# average those nearest-neighbor distances into a single summary distance.
def avg_distance_to_set(gene_coords, target_coords):
    distances = cdist(gene_coords, target_coords, metric='euclidean')
    min_distances = np.min(distances, axis=1)
    return np.mean(min_distances)

# Coordinates for the known/reference gene set, and for all genes (used to
# draw the random samples for the permutation null distribution).
known_coords = df[df['gene'].isin(known_genes)][['x','y']].values
all_coords = df[['x','y']].values

# Permutation test parameters
n_permutations = 1000
np.random.seed(42)

results = []

for gene in test_genes:
    # Coordinates for this test gene.
    gene_coords = df[df['gene'] == gene][['x','y']].values
    if gene_coords.shape[0] == 0:
        print(f"⚠️ No coordinates found for {gene}, skipping.")
        continue

    # Observed average nearest-neighbor distance to the known-gene set.
    observed_distance = avg_distance_to_set(gene_coords, known_coords)

    # Build the null distribution by repeatedly substituting a random sample
    # of points (same count as the test gene) for the test gene's coordinates.
    permuted_distances = []
    n_points = gene_coords.shape[0]

    for _ in range(n_permutations):
        random_coords = all_coords[np.random.choice(all_coords.shape[0], n_points, replace=False)]
        permuted_distances.append(avg_distance_to_set(random_coords, known_coords))

    permuted_distances = np.array(permuted_distances)

    # One-sided empirical p-value: fraction of random distances at least as
    # small as the observed (real) distance.
    p_value = np.sum(permuted_distances <= observed_distance) / n_permutations

    results.append((gene, observed_distance, p_value))

# Print results
for gene, dist, pval in results:
    print(f"Observed avg distance of {gene} to known genes: {dist:.3f}")
    print(f"P-value for being close: {pval:.4f}\n")


### 2.3 UMAP Scatter Plot with Highlighted Genes

Plots every gene's UMAP coordinate as a light background point, then overlays the `known_genes` (blue) and `test_genes` (red) as larger, labeled markers. `adjust_text` is used to automatically reposition overlapping text labels and draw thin leader lines back to their points.


In [ ]:
# --- Prepare dataframe ---
df = df.rename(columns={"Unnamed: 0": "gene"})

# --- Genes to highlight in the lncHUB2 UMAP plot ---
highlight_genes = known_genes + test_genes

# --- Background scatter of all genes ---
plt.figure(figsize=(8, 7))
plt.scatter(df['x'], df['y'], color='lightblue', alpha=0.8, s=15)

texts = []

# --- Overlay known genes (blue) ---
for _, row in df.iterrows():
    if row['gene'] in known_genes:
        plt.scatter(row['x'], row['y'], color='blue', s=150, zorder=3)
        texts.append(
            plt.text(
                row['x'], row['y'], row['gene'],
                fontsize=14, fontweight='bold', color='blue',
                ha='center', va='center'
            )
        )

# --- Overlay test genes (red) ---
for _, row in df.iterrows():
    if row['gene'] in test_genes:
        plt.scatter(row['x'], row['y'], color='red', s=150, zorder=3)
        texts.append(
            plt.text(
                row['x'], row['y'], row['gene'],
                fontsize=14, fontweight='bold', color='red',
                ha='center', va='center'
            )
        )

# --- Automatically adjust text to avoid overlap ---
adjust_text(
    texts,
    arrowprops=dict(arrowstyle='-', color='gray', alpha=0.5),
    expand_points=(3, 3),    # more space around points
    expand_text=(3, 3),      # more space around labels
    force_points=3,          # stronger push away from points
    force_text=3,            # stronger repulsion between labels
    lim=500,                 # more iterations for better layout
)

# --- Labels and formatting ---
plt.xlabel('X', fontsize=14)
plt.ylabel('Y', fontsize=14)
plt.title(
    'Gene Coordinates Highlighting Known (blue) and Test (red) Genes',
    fontsize=16, fontweight='bold'
)
plt.grid(True)
plt.tight_layout()
plt.show()


## 3. Tissue and Cell-Type Expression Databases


### 3.1 Load Tissue and Cell-Line Expression Tables

Both files contain per-gene summary statistics (e.g. min/quartiles/median/max) of expression across many tissues or cell lines respectively, as distributed by lncHUB2.


In [ ]:
# --------------------------
# Load CSVs
# --------------------------
tissue_expr = pd.read_csv('../../data/lncHUB2/tissue_expr.zip')
cell_line_expr = pd.read_csv('../../data/lncHUB2/cell_line_expr.zip')


### 3.2 Configure Highlighted Tissues / Cell Lines

`highlight_tissues` and `highlight_cell_lines` name specific tissues/cell lines to bold and color red on the y-axis of the plots below. Replace `"."` with the actual tissue/cell-line label(s) of interest (must match the identifiers used in the source tables).


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Tissue-specific highlights
highlight_tissues = ["."]  # USER NEEDS TO HIGHLIGHT TISSUES OF INTEREST

# Cell line-specific highlights
highlight_cell_lines = ["."]  # USER NEEDS TO HIGHLIGHT CELL TYPES OF INTEREST


### 3.3 Plot Expression Distributions per Gene

`plot_expression` draws a 2x2 grid of manual box-and-whisker plots, one panel per gene in `test_genes`:

1. For each gene, the matching row is located (by `Unnamed: 0`, the gene-ID column) and re-indexed by `Unnamed: 1`, the summary-statistic name (`min`/`25%`/`50%`/`75%`/`max`).
2. Columns are converted to numeric, and only the **top 10** tissues/cell lines by median expression are kept.
3. Values are log10-transformed (`log10(x + 1)`) for display.
4. A box-and-whisker shape is drawn manually for each of the 10 items: a line spanning min-to-max, a shaded rectangle spanning the interquartile range (25th-75th percentile), and a red dot marking the median.
5. Y-axis tick labels matching `highlight_items` are bolded and colored red.

The function is then called once for the tissue table and once for the cell-line table.


In [ ]:
# --------------------------
# Function to plot expression
# --------------------------
def plot_expression(df, title_prefix, highlight_items):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))  # 2x2 grid, one panel per gene
    axes = axes.flatten()

    for idx, gene in enumerate(test_genes):
        ax = axes[idx]

        # Select the row(s) for this gene and re-index by statistic name.
        gene_stats = df[df['Unnamed: 0'] == gene].set_index('Unnamed: 1')
        gene_stats = gene_stats.drop(columns=['Unnamed: 0'])
        gene_stats = gene_stats.apply(pd.to_numeric)

        # Keep only the top 10 tissues/cell lines by median expression.
        top10 = gene_stats.loc[:, gene_stats.loc['50%'].sort_values(ascending=False).index[:10]]

        # Log-transform for display.
        top10 = np.log10(top10 + 1)

        # Extract the five summary statistics needed for the box plot.
        items = top10.columns
        min_vals = top10.loc['min']
        q1 = top10.loc['25%']
        median = top10.loc['50%']
        q3 = top10.loc['75%']
        max_vals = top10.loc['max']

        # Manually draw a box-and-whisker shape for each item (one per row).
        for i, (min_val, q1_val, med_val, q3_val, max_val) in enumerate(zip(min_vals, q1, median, q3, max_vals)):
            ax.plot([min_val, max_val], [i, i], color='black', linewidth=1)  # whisker: min-max
            ax.add_patch(plt.Rectangle((q1_val, i-0.3), q3_val-q1_val, 0.6,
                                       facecolor='lightblue', edgecolor='black'))  # IQR box
            ax.plot(med_val, i, 'ro')  # median marker

        # Customize y-axis
        ax.set_yticks(np.arange(len(items)))
        ax.set_yticklabels(items, fontsize=12)
        ax.invert_yaxis()
        ax.set_xlabel('log10(Expression + 1)')
        ax.set_title(f"{gene}", fontsize=14, fontweight='bold')

        # Bold/color any tick labels matching the configured highlight list.
        for tick_label in ax.get_yticklabels():
            if tick_label.get_text() in highlight_items:
                tick_label.set_color('red')
                tick_label.set_fontweight('bold')

    # Hide any unused subplot panels (if fewer than 4 test genes).
    for j in range(len(test_genes), len(axes)):
        axes[j].axis('off')

    # Main figure title
    fig.suptitle(f"{title_prefix}", fontsize=20, fontweight='bold', y=0.95)

    # Adjust layout
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()


# --------------------------
# Plot tissue expression
# --------------------------
plot_expression(tissue_expr,
                title_prefix="Expression of CLL lncRNAs of interest in Tissues",
                highlight_items=highlight_tissues)

# --------------------------
# Plot cell line expression
# --------------------------
plot_expression(cell_line_expr,
                title_prefix="Expression of CLL lncRNAs of interest in Cell Lines",
                highlight_items=highlight_cell_lines)
